# Avance Fase 3 — Semana 2

## Núcleo algorítmico, eficiencia y programación orientada a objetos

**Proyecto:** Uso de redes sociales y salud mental autopercibida en estudiantes de enseñanza media (YRBS 2023, CDC)
**Asignatura:** MCDI500 · Programación para la Ciencia de Datos · **Grupo 8** · Formativa 3
**Integrantes:** Matías Manríquez Ortiz · Daniel Pérez Ramirez · Abigail Robles Chávez · Roberto Sánchez Saldivia

---

En la Fase 2 construimos un pipeline que deja el YRBS 2023 listo para el análisis
(`F2/data/processed/yrbs2023_seleccion_procesada.csv`). En este cuaderno tomamos ese resultado y
le damos forma de **sistema**: las mismas operaciones, ahora organizadas en clases que se pueden
probar, reemplazar y reutilizar.

No hay datos nuevos ni pipeline nuevo. **Lo que cambia es la arquitectura del código.**

> **Decisiones de la Fase 2 que se mantienen.** Los faltantes se conservan como NA (bitácora 2.5)
> y las escalas ordinales no se escalan (sección 5 de F2). Los ejemplos de imputación y escalamiento
> de las partes 1 a 11 son **didácticos**: muestran cómo funcionan las clases sobre una columna. El
> pipeline de las partes 12 y 13 respeta las decisiones del proyecto.

### Recorrido

| Parte | Contenido | Criterio que evidencia |
|---|---|---|
| 1 | De funciones a objetos: por qué | Codificación funcional |
| 2 | Los cinco conceptos de la POO | Programación orientada a objetos |
| 3 | La clase `Preprocesador` | Preprocesamiento y transformación |
| 4 | Encapsulamiento: proteger el estado | Programación orientada a objetos |
| 5 | Herencia y polimorfismo | Programación orientada a objetos |
| 6 | El `Pipeline`: componer los pasos | Diseño estructurado |
| 7 | Cohesión y acoplamiento | Diseño estructurado |
| 8 | Validación: casos normales, límite y excepciones | Validación técnica |
| 9 | Recursividad | Diseño estructurado |
| 10 | Eficiencia: medir tiempo y memoria | Eficiencia y optimización |
| 11 | Patrones de diseño | Documentación de arquitectura |
| 12 | Pipeline construido desde la configuración | Codificación funcional |
| 13 | El resultado: código y datos del pipeline | Documentación de arquitectura |

**Cómo se usa.** Los datos se leen **desde fuera**: la celda de configuración localiza la raíz
del repositorio, igual que en F1 y F2, y desde ahí lee el archivo de la Fase 2.

---
## Configuración del entorno y de las rutas

Igual que en F1 y F2, las rutas se declaran **relativas a la raíz del repositorio**, que se
localiza subiendo desde la carpeta del cuaderno hasta encontrar `.git`. Así el cuaderno corre
en cualquier computador del grupo sin editar rutas. Esta celda concentra rutas, columnas y
decisiones; el resto del cuaderno se construye a partir de ella.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# Rutas, columnas y decisiones del conjunto de trabajo
# =====================================================================
import sys
from pathlib import Path


def encontrar_raiz(desde=None):
    '''Sube desde la carpeta de trabajo hasta la raíz del repositorio (carpeta con .git).'''
    actual = Path(desde or Path.cwd()).resolve()
    for carpeta in (actual, *actual.parents):
        if (carpeta / ".git").exists():
            return carpeta
    raise FileNotFoundError(f"No se encontró el repositorio sobre {actual}.")


RAIZ = encontrar_raiz()
RUTA_DATOS = RAIZ / 'F2' / 'data' / 'processed' / 'yrbs2023_seleccion_procesada.csv'   # salida de F2
DIR_DEMO = RAIZ / 'F3' / 'data' / '_demo'                                            # salidas de este cuaderno

COLUMNA_OBJETIVO = "salud_mental_cod"          # la variable que se quiere explicar
COLUMNA_ID = "id_registro"                    # identificador: se elimina del análisis

# Escalas ordinales del codebook (códigos, no cantidades)
COLUMNAS_ORDINALES = ["redes_sociales_cod", "sueno_cod", "actividad_fisica_cod", "edad_cod"]
COLUMNAS_NOMINALES = ["raceeth_cod", "sexo_cod"]

SEMILLA = 42
PROPORCION_PRUEBA = 0.2

# Decisiones de la Fase 2 que gobiernan el pipeline de las partes 12 y 13
IMPUTAR_ORDINALES = False    # bitácora 2.5: los faltantes se conservan como NA
ESCALAR_ORDINALES = False    # F2, sección 5: las escalas ordinales no se escalan

# Columnas usadas en los ejemplos didácticos del cuaderno
COLUMNA_IMPUTAR = "actividad_fisica_cod"   # con faltantes (6,1 %): ejemplos de imputación
COLUMNA_ESCALAR = "edad_cod"               # ejemplos de escalamiento
COLUMNA_GRUPO = "sexo_cod"                 # imputación por grupo (patrón Strategy)
COLUMNA_CLASIFICAR = "sueno_cod"           # comparación bucle vs. vectorizado
# =====================================================================

COLUMNAS_ESPERADAS = ([COLUMNA_ID, COLUMNA_OBJETIVO] + COLUMNAS_ORDINALES
                      + COLUMNAS_NOMINALES)

print("Raíz              :", RAIZ)
print("Datos de F2       :", RUTA_DATOS.relative_to(RAIZ).as_posix(), "·",
      "existe" if RUTA_DATOS.exists() else "NO EXISTE")
print("Salidas           :", DIR_DEMO.relative_to(RAIZ).as_posix())
print("Variable objetivo :", COLUMNA_OBJETIVO)
print("Columnas esperadas:", len(COLUMNAS_ESPERADAS))

Raíz              : C:\Users\danie\OneDrive\Universidad Andres Bello\Magister Cienca Datos e IA\Semestre 1\202609 Septiembre\MCDI500 Programación para la Ciencia de Datos\GitHub\proyecto-grupo8-mcdi500
Datos de F2       : F2/data/processed/yrbs2023_seleccion_procesada.csv · existe
Salidas           : F3/data/_demo
Variable objetivo : salud_mental_cod
Columnas esperadas: 8


## Preparación del entorno

In [2]:
import os
import time
import timeit
import tracemalloc

import numpy as np
import pandas as pd

np.random.seed(SEMILLA)

print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)

pandas : 3.0.6
NumPy  : 2.5.3
Semilla: 42


## Carga del conjunto desde el archivo externo

El cuaderno **no genera datos**: lee el archivo que dejó la Fase 2. La función de carga hace
tres cosas antes de devolver el conjunto:

1. Comprueba que el archivo exista y, si no, indica qué ruta revisar.
2. Lo lee según su extensión (CSV, Excel o Parquet).
3. Verifica que estén las columnas declaradas en la configuración.

El tercer paso evita descubrir tarde, varias celdas más abajo, que una columna cambió de nombre.

In [3]:
def leer_archivo(ruta):
    """Lee el archivo según su extensión y devuelve un DataFrame."""
    extension = os.path.splitext(ruta)[1].lower()

    if extension in (".csv", ".txt"):
        # sep=None con engine="python" infiere el separador: útil con datos
        # públicos chilenos, que suelen venir con punto y coma
        return pd.read_csv(ruta, sep=None, engine="python", encoding="utf-8")
    if extension in (".xlsx", ".xls"):
        return pd.read_excel(ruta)
    if extension == ".parquet":
        return pd.read_parquet(ruta)

    raise ValueError(
        f"Extensión no reconocida: '{extension}'. "
        "Se admiten .csv, .txt, .xlsx, .xls y .parquet."
    )


def verificar_esquema(df, columnas_esperadas):
    """Comprueba que estén todas las columnas declaradas en la configuración."""
    faltantes = [c for c in columnas_esperadas if c not in df.columns]
    if faltantes:
        raise KeyError(
            f"Faltan columnas declaradas en la configuración: {faltantes}\n"
            f"Columnas disponibles en el archivo: {list(df.columns)}"
        )
    sobrantes = [c for c in df.columns if c not in columnas_esperadas]
    return {"declaradas": len(columnas_esperadas),
            "en_archivo": df.shape[1],
            "no_declaradas": sobrantes}

In [4]:
def cargar(ruta=RUTA_DATOS, columnas_esperadas=None):
    """Carga el conjunto desde el archivo externo y verifica su esquema."""
    columnas_esperadas = columnas_esperadas or COLUMNAS_ESPERADAS

    if not os.path.exists(ruta):
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}\n"
            f"Carpeta actual: {os.getcwd()}\n"
            "Revisar RUTA_DATOS en la celda de configuración: la ruta se arma "
            "desde la raíz del repositorio."
        )
    df = leer_archivo(ruta)
    print(f"Archivo leído: {Path(ruta).relative_to(RAIZ).as_posix()}")

    # Las columnas de códigos se fuerzan a número: un texto suelto las
    # convertiría en columna de objetos sin ningún aviso
    for columna in COLUMNAS_ORDINALES:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors="coerce")

    informe = verificar_esquema(df, columnas_esperadas)
    print(f"Forma: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"Esquema verificado: {informe['declaradas']} columnas declaradas, "
          f"{len(informe['no_declaradas'])} no declaradas")
    if informe["no_declaradas"]:
        print("  No declaradas:", informe["no_declaradas"])
    return df


datos = cargar()
datos.head()

Archivo leído: F2/data/processed/yrbs2023_seleccion_procesada.csv
Forma: 20103 filas x 26 columnas
Esquema verificado: 8 columnas declaradas, 18 no declaradas
  No declaradas: ['peso_muestral', 'estrato', 'psu', 'n_faltantes_analisis', 'caso_completo', 'raza_amerindia', 'raza_asiatica', 'raza_negra', 'raza_hawaiana_pacifico', 'raza_blanca', 'raza_hispana', 'raza_multiple_hispana', 'raza_multiple_no_hispana', 'sexo_femenino', 'salud_mental_mala', 'redes_uso_frecuente', 'sueno_8h_o_mas', 'actividad_5_dias']


,id_registro,peso_muestral,estrato,psu,edad_cod,sexo_cod,raceeth_cod,salud_mental_cod,redes_sociales_cod,sueno_cod,...,raza_hawaiana_pacifico,raza_blanca,raza_hispana,raza_multiple_hispana,raza_multiple_no_hispana,sexo_femenino,salud_mental_mala,redes_uso_frecuente,sueno_8h_o_mas,actividad_5_dias
0,1,0.8614,103,16294,3.0,1.0,NaN,1.0,6.0,3.0,...,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1.0,0.0,0.0
1,2,0.8920,103,16294,4.0,2.0,5.0,3.0,4.0,5.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,3,0.5081,103,16294,5.0,2.0,5.0,2.0,8.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,4,1.1759,103,16294,6.0,1.0,5.0,3.0,8.0,4.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,5,0.8920,103,16294,3.0,2.0,5.0,3.0,6.0,3.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


**Si aparece un `KeyError` al cargar**, la configuración no coincide con el archivo. El mensaje
lista las columnas que faltan y las disponibles, así que basta con corregir los nombres en la
configuración y volver a ejecutar.

## El conjunto antes de tocarlo

Perfilamos el conjunto con una función genérica: se apoya en la configuración, así que sirve
igual si cambian las columnas.

In [5]:
def perfilar(df):
    """Devuelve una fila por columna con su tipo, únicos y porcentaje de nulos."""
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).sort_values("pct_nulos", ascending=False).reset_index(drop=True)


perfil = perfilar(datos)
print("Columnas con valores faltantes:")
print(perfil[perfil["nulos"] > 0].to_string(index=False))

print(f"\nDistribución de la variable objetivo '{COLUMNA_OBJETIVO}':")
print(datos[COLUMNA_OBJETIVO].value_counts(normalize=True).round(4))

for columna in COLUMNAS_NOMINALES:
    print(f"\nCategorías de {columna}:")
    print(datos[columna].value_counts(dropna=False).sort_index())

Columnas con valores faltantes:
                 columna    tipo  unicos  nulos  pct_nulos
      redes_sociales_cod float64       8   4900      24.37
     redes_uso_frecuente float64       2   4900      24.37
       salud_mental_mala float64       2   4398      21.88
        salud_mental_cod float64       5   4398      21.88
               sueno_cod float64       7   2662      13.24
          sueno_8h_o_mas float64       2   2662      13.24
    actividad_fisica_cod float64       8   1227       6.10
        actividad_5_dias float64       2   1227       6.10
          raza_amerindia float64       2    370       1.84
             raceeth_cod float64       8    370       1.84
   raza_multiple_hispana float64       2    370       1.84
raza_multiple_no_hispana float64       2    370       1.84
             raza_blanca float64       2    370       1.84
  raza_hawaiana_pacifico float64       2    370       1.84
            raza_hispana float64       2    370       1.84
           raza_asiatica

**Tres observaciones que condicionan todo lo que viene (YRBS 2023).**

1. **Hay faltantes reales en todas las variables de análisis**, entre 0,5 % (`edad_cod`) y 24,4 %
   (`redes_sociales_cod`). En la Fase 2 se mostró que no son aleatorios y se decidió **no imputarlos**
   (bitácora 2.5). Aquí la imputación aparece solo como ejemplo didáctico sobre `COLUMNA_IMPUTAR`.
2. **Los códigos son ordinales o nominales, no cantidades.** `sexo_cod` y `raceeth_cod` son nominales:
   un código 6 no «vale más» que un 2. Por eso se codifican con columnas 0/1.
3. **La variable objetivo (`salud_mental_cod`) tiene 21,9 % de faltantes.** No se imputa: la separación
   entrenamiento/prueba se hace solo con los registros que la respondieron (parte 4).

---
# 1. De funciones a objetos: por qué

Así quedó el código de la Fase 2, escrito en celdas sueltas:

In [6]:
# El estilo de la Fase 2: funciona, pero cada celda depende de las anteriores
df_f2 = datos.copy()
df_f2 = df_f2.drop(columns=[COLUMNA_ID])
df_f2 = pd.get_dummies(df_f2, columns=["raceeth_cod"], dtype=int)

print("Resultado:", df_f2.shape)


Resultado: (20103, 32)


Funciona, pero tiene cuatro problemas que aparecen cuando el proyecto crece:

| Problema | Consecuencia |
|---|---|
| El orden importa y no está declarado | Ejecutar una celda fuera de orden produce otro resultado |
| No se puede reutilizar sin copiar | El mismo código se duplica en el cuaderno de F3 y F4 |
| No se puede probar por partes | Si el resultado está mal, hay que revisar todo |
| Los parámetros se calculan sobre todo el conjunto | Si después se separan entrenamiento y prueba, hay **fuga de datos** |

La programación orientada a objetos resuelve los cuatro: cada paso pasa a ser una pieza
con nombre, con estado propio y con una interfaz fija.

---
# 2. Programación orientada a objetos desde cero

Esta parte deja por escrito los conceptos base de la POO que usamos en el resto del
cuaderno. El trazado de la sección 2.3 muestra qué es `self` y por qué todo método lo
recibe.


## 2.1 Clase y objeto

Una **clase** es un molde: describe qué datos guarda algo y qué operaciones sabe hacer. Un
**objeto** es una copia concreta hecha con ese molde; de un mismo molde salen muchas copias y
cada una guarda sus propios valores.

Lo pensamos como un formulario: la clase es el formulario en blanco, que define los campos, y
cada objeto es un formulario llenado con sus propios valores.

En el proyecto usamos una clase cuando un componente necesita **recordar algo** entre una
llamada y otra; si no necesita recordar nada, basta una función. Ese es nuestro criterio para
decidir entre función y clase.

In [7]:
# Clase mínima con los cinco conceptos: clase, objeto, atributo, método y self
class Contador:
    """Cuenta cuántas veces se le pide contar.

    El docstring, este texto entre comillas triples, es la documentación de la
    clase. Aparece al escribir help(Contador).
    """

    def __init__(self):
        # __init__ es el CONSTRUCTOR: Python lo ejecuta solo al crear el objeto.
        # Su trabajo es dejar listos los atributos.
        # self es el objeto que se está creando.
        self.total = 0          # ATRIBUTO: un dato que el objeto guarda

    def sumar(self, cantidad=1):
        # MÉTODO: una operación que el objeto sabe hacer.
        # Recibe self porque necesita saber sobre CUÁL objeto opera.
        self.total = self.total + cantidad
        return self.total


# Al crear el objeto, Python ejecuta __init__ automáticamente
c = Contador()
print("Al crearlo, total vale:", c.total)

c.sumar()        # sin argumento: usa cantidad=1 por defecto
c.sumar(5)
print("Después de sumar 1 y 5:", c.total)
print("Tipo del objeto:", type(c))

Al crearlo, total vale: 0
Después de sumar 1 y 5: 6
Tipo del objeto: <class '__main__.Contador'>


## 2.2 Cada objeto tiene sus propios datos

Dos objetos de la misma clase **no comparten** sus atributos de instancia: cada uno guarda los
suyos.

In [8]:
c1 = Contador()          # primer formulario llenado
c2 = Contador()          # segundo formulario, independiente del primero

c1.sumar(10)
c2.sumar(3)

print("c1.total:", c1.total)
print("c2.total:", c2.total)
print("¿Son el mismo objeto?", c1 is c2)
print("\nCada objeto guarda lo suyo. El molde es el mismo; los datos, no.")

# Lo que cada objeto guarda se puede ver directamente
print("\nContenido interno de c1:", c1.__dict__)
print("Contenido interno de c2:", c2.__dict__)

c1.total: 10
c2.total: 3
¿Son el mismo objeto? False

Cada objeto guarda lo suyo. El molde es el mismo; los datos, no.

Contenido interno de c1: {'total': 10}
Contenido interno de c2: {'total': 3}


## 2.3 Qué es `self`

`self` responde a una pregunta práctica: si el método está escrito una sola vez en la clase,
**¿cómo sabe sobre qué objeto trabajar?**

Python le pasa el objeto como primer argumento. Cuando se escribe `c1.sumar(10)`, Python
ejecuta por detrás `Contador.sumar(c1, 10)`. Por eso `self` no tiene nada de especial: es el
primer parámetro, y se llama así por convención.

In [9]:
c3 = Contador()

# Las dos líneas siguientes hacen exactamente lo mismo
c3.sumar(7)                 # forma normal
Contador.sumar(c3, 7)       # lo que Python ejecuta por detrás

print("Total tras las dos llamadas equivalentes:", c3.total)
print("\nPor eso todo método lleva self como primer parámetro:")
print("es el espacio donde llega el objeto sobre el que se opera.")

Total tras las dos llamadas equivalentes: 14

Por eso todo método lleva self como primer parámetro:
es el espacio donde llega el objeto sobre el que se opera.


## 2.4 Dónde busca Python un atributo

Cuando se escribe `objeto.algo`, Python lo busca en este orden:

1. En el **objeto**, es decir en su `__dict__`.
2. Si no está, en su **clase**.
3. Si no está, en la **clase madre**, y así hacia arriba.

Esa cadena se llama MRO (*method resolution order*) y se puede consultar. Es la que explica por
qué funciona la herencia de la parte 5.

In [10]:
class Persona:
    especie = "humano"                  # atributo de CLASE: uno solo, compartido

    def __init__(self, nombre):
        self.nombre = nombre            # atributo de INSTANCIA: uno por objeto


ana = Persona("Ana")
luis = Persona("Luis")

print("Atributos propios de ana :", ana.__dict__)
print("Atributo de la clase     :", Persona.especie)
print("ana.especie ->", ana.especie, "(no está en ana: Python sube a la clase)")

# Cambiar el atributo de clase afecta a TODOS los objetos
Persona.especie = "homo sapiens"
print("\nDespués de cambiarlo en la clase:")
print("  ana.especie :", ana.especie)
print("  luis.especie:", luis.especie)

# Cambiar un atributo de instancia afecta solo a ese objeto
ana.nombre = "Ana María"
print("\nDespués de cambiar solo ana.nombre:")
print("  ana.nombre :", ana.nombre)
print("  luis.nombre:", luis.nombre)

print("\nCadena de búsqueda (MRO):", [c.__name__ for c in Persona.__mro__])

Atributos propios de ana : {'nombre': 'Ana'}
Atributo de la clase     : humano
ana.especie -> humano (no está en ana: Python sube a la clase)

Después de cambiarlo en la clase:
  ana.especie : homo sapiens
  luis.especie: homo sapiens

Después de cambiar solo ana.nombre:
  ana.nombre : Ana María
  luis.nombre: Luis

Cadena de búsqueda (MRO): ['Persona', 'object']


**En la práctica.** Un atributo de clase sirve para constantes compartidas, como una etiqueta o
un valor por defecto. Un atributo de instancia guarda lo que cambia de un objeto a otro, que es
casi todo lo que necesita un pipeline.

## 2.5 Resumen de los cinco conceptos

| Concepto | Qué es | Cómo se reconoce en el código |
|---|---|---|
| **Clase** | El molde | `class NombreDeLaClase:` |
| **Objeto** | Una copia del molde | `obj = NombreDeLaClase(...)` |
| **Atributo** | Un dato que el objeto guarda | `self.algo = valor` |
| **Método** | Una operación del objeto | `def hacer(self, ...):` |
| **`self`** | El objeto sobre el que se opera | Primer parámetro de todo método |

Con estos cinco conceptos se lee el resto del cuaderno.

---
# 3. La clase `Preprocesador`

Primera versión: una clase que guarda la tabla y sabe prepararla. Cada método hace
**una sola cosa** y devuelve `self`, lo que permite encadenar llamadas.

In [11]:
class Preprocesador:
    """Guarda la tabla del proyecto y sabe prepararla para el análisis.

    La tabla vive en self.df. Cada método es un paso del pipeline que trabaja
    sobre esa misma tabla y devuelve self, para poder encadenar.
    """

    def __init__(self, df):
        self.df = df.copy()          # copia: no se modifica el original
        self.registro = []           # lo que el objeto va recordando

    def quitar_identificador(self):
        """El id no aporta información al análisis: identifica, no describe."""
        if COLUMNA_ID in self.df.columns:
            self.df = self.df.drop(columns=[COLUMNA_ID])
            self.registro.append(f"{COLUMNA_ID} eliminado")
        return self

    def imputar(self, columna):
        """Rellena los nulos de una columna con la mediana."""
        nulos = int(self.df[columna].isna().sum())
        mediana = float(self.df[columna].median())
        self.df[columna] = self.df[columna].fillna(mediana)
        self.registro.append(f"{columna}: {nulos} nulos imputados con mediana={mediana:.1f}")
        return self

    def codificar(self, columnas):
        """Convierte columnas de categorías en columnas 0/1."""
        antes = self.df.shape[1]
        self.df = pd.get_dummies(self.df, columns=columnas, dtype=int)
        self.registro.append(f"codificadas {columnas}: {antes} -> {self.df.shape[1]} columnas")
        return self

    def informe(self):
        """Devuelve lo que el objeto recuerda haber hecho."""
        return "\n".join(f"  {i}. {paso}" for i, paso in enumerate(self.registro, 1))


prep = Preprocesador(datos)
prep.quitar_identificador().imputar(COLUMNA_IMPUTAR).codificar(COLUMNAS_NOMINALES)

print("Resultado:", prep.df.shape)
print("\nLo que el objeto recuerda:")
print(prep.informe())

Resultado: (20103, 33)

Lo que el objeto recuerda:
  1. id_registro eliminado
  2. actividad_fisica_cod: 1227 nulos imputados con mediana=5.0
  3. codificadas ['raceeth_cod', 'sexo_cod']: 25 -> 33 columnas


**Lo que ganamos.** El objeto `prep` recuerda qué hizo y con qué valores. En la Fase 2 esa
información la anotamos a mano en la bitácora; aquí sale de la propia ejecución y no puede
quedar desactualizada.

**Lo que falta.** Esta clase mezcla dos cosas: aprender de los datos y aplicar lo aprendido.
Eso lo resolvemos en la parte siguiente.

---
# 4. Encapsulamiento: proteger el estado interno

El `Preprocesador` tiene un problema: calcula la mediana **sobre el conjunto completo**. Si
después se separan entrenamiento y prueba, la mediana ya vio los datos de prueba. Eso es
**fuga de datos**, uno de los errores más costosos en ciencia de datos.

Lo resolvemos separando dos momentos:

- **`ajustar`**: aprende los parámetros, solo del conjunto de entrenamiento.
- **`transformar`**: los aplica a cualquier conjunto.

Y protegemos ese estado interno con un guion bajo al inicio del nombre, que en Python indica
que el atributo es interno y no se modifica desde fuera.

In [12]:
class ImputadorSimple:
    """Imputa una columna separando lo que aprende de lo que aplica (ejemplo didáctico)."""

    def __init__(self, columna=COLUMNA_IMPUTAR):
        self.columna = columna
        self._mediana = None        # interno: se aprende, no se asigna desde fuera
        self._ajustado = False      # interno: controla el orden de las llamadas

    @property
    def ajustado(self):
        """Solo lectura: se puede consultar, no asignar."""
        return self._ajustado

    @property
    def mediana(self):
        return self._mediana

    def ajustar(self, df):
        self._mediana = float(df[self.columna].median())
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError(
                "ImputadorSimple: hay que llamar a ajustar() antes de transformar(). "
                "La mediana se aprende del conjunto de entrenamiento."
            )
        df = df.copy()
        df[self.columna] = df[self.columna].fillna(self._mediana)
        return df


# El orden equivocado ahora falla con un mensaje claro
imputador = ImputadorSimple()
try:
    imputador.transformar(datos)
except RuntimeError as error:
    print("RuntimeError:", error)

RuntimeError: ImputadorSimple: hay que llamar a ajustar() antes de transformar(). La mediana se aprende del conjunto de entrenamiento.


In [13]:
# El orden correcto, con separación de conjuntos
# La variable objetivo no se imputa (bitácora 2.5): solo se separan los registros
# que la respondieron
datos_objetivo = datos.dropna(subset=[COLUMNA_OBJETIVO])
print(f"Registros con {COLUMNA_OBJETIVO} respondida: {len(datos_objetivo)} de {len(datos)}")

entrenamiento = datos_objetivo.sample(frac=1 - PROPORCION_PRUEBA, random_state=SEMILLA)
prueba = datos_objetivo.drop(index=entrenamiento.index)

imputador = ImputadorSimple().ajustar(entrenamiento)
prueba_lista = imputador.transformar(prueba)

print("Mediana aprendida del entrenamiento:", round(imputador.mediana, 2))
print("Mediana del conjunto de prueba     :", round(prueba[COLUMNA_IMPUTAR].median(), 2))
print("Nulos en prueba después de imputar :", int(prueba_lista[COLUMNA_IMPUTAR].isna().sum()))
if imputador.mediana != prueba[COLUMNA_IMPUTAR].median():
    print("\nSon medianas distintas, y está bien: se usó la del entrenamiento.")
else:
    print("\nAquí coinciden porque el código es discreto (1 a 8), pero el valor usado es el del entrenamiento.")

# La propiedad es de solo lectura
try:
    imputador.ajustado = False
except AttributeError as error:
    print("\nAttributeError:", error)

Registros con salud_mental_cod respondida: 15705 de 20103
Mediana aprendida del entrenamiento: 5.0
Mediana del conjunto de prueba     : 5.0
Nulos en prueba después de imputar : 0

Aquí coinciden porque el código es discreto (1 a 8), pero el valor usado es el del entrenamiento.

AttributeError: property 'ajustado' of 'ImputadorSimple' object has no setter


**Por qué importa.** Sin el control de `_ajustado`, transformar antes de ajustar produciría un
resultado sin sentido **y sin ningún mensaje de error**. El encapsulamiento convierte ese error
silencioso en uno visible.

En el informe usamos este bloque como evidencia del encapsulamiento: atributos internos con
guion bajo, propiedades de solo lectura y validación antes de actuar.

---
# 5. Herencia y polimorfismo

En el `ImputadorSimple`, el control de `_ajustado`, la copia defensiva y el mensaje de error se
repetirían en el codificador, en el escalador y en cualquier otro paso.

**Herencia** es escribir eso una sola vez en una clase base.

In [14]:
class Transformador:
    """Clase base: lo que todos los pasos del pipeline comparten.

    Esta clase NO se usa directamente. Su trabajo es definir el contrato:
    qué métodos tendrá todo paso del pipeline y cómo se controla su estado.
    Las clases hijas solo completan lo que cambia de un paso a otro.
    """

    def __init__(self, columna):
        # Atributo público: cualquiera puede leerlo y cambiarlo
        self.columna = columna

        # Atributos con guion bajo: por convención, INTERNOS.
        # Python no lo impide, pero el guion bajo le dice a quien lee el código
        # que no debe tocarlos desde fuera de la clase.
        self._parametros = {}       # lo que el paso aprende del conjunto
        self._ajustado = False      # controla que no se transforme antes de ajustar

    # @property convierte un método en algo que se lee como atributo:
    # se escribe paso.nombre y no paso.nombre(). Y como no hay setter,
    # el valor NO se puede asignar desde fuera. Eso es encapsulamiento.
    @property
    def nombre(self):
        # type(self).__name__ devuelve el nombre de la clase REAL del objeto,
        # así que un ImputadorMediana se identifica como tal y no como Transformador
        return f"{type(self).__name__}({self.columna})"

    @property
    def parametros(self):
        # dict(...) crea una COPIA. Si devolviéramos self._parametros
        # directamente, quien lo recibe podría modificar el estado interno
        # del objeto sin que la clase se entere. A esto se le llama
        # copia defensiva.
        return dict(self._parametros)

    def ajustar(self, df):
        """Aprende los parámetros del conjunto que recibe."""
        # 1) Validar la entrada ANTES de trabajar: si algo falta, se avisa
        #    aquí y no más abajo con un error difícil de rastrear
        if self.columna not in df.columns:
            raise KeyError(f"{self.nombre}: la columna no existe en el conjunto.")

        # 2) Delegar el cálculo a la clase hija. La base no sabe QUÉ se aprende;
        #    solo sabe CUÁNDO hay que aprenderlo.
        self._parametros = self.aprender(df)

        # 3) Registrar que ya se ajustó, para poder controlarlo después
        self._ajustado = True

        # 4) Devolver self permite encadenar: paso.ajustar(df).transformar(df)
        return self

    def transformar(self, df):
        """Aplica la transformación usando lo aprendido en ajustar()."""
        # Esta guarda es el corazón del encapsulamiento: impide un uso
        # incorrecto que de otro modo pasaría inadvertido
        if not self._ajustado:
            raise RuntimeError(f"{self.nombre}: hay que ajustar antes de transformar.")

        # df.copy() evita modificar el DataFrame que nos pasaron. Sin esto,
        # el DataFrame original cambiaría sin que nadie lo pidiera.
        return self.aplicar(df.copy())

    def ajustar_transformar(self, df):
        """Atajo para el caso habitual: aprender y aplicar sobre el mismo conjunto."""
        return self.ajustar(df).transformar(df)

    # ---- Métodos que cada clase hija DEBE implementar -------------------
    # Se definen aquí lanzando NotImplementedError para dejar el contrato
    # explícito: si una hija olvida implementarlos, el error lo dice claro.

    def aprender(self, df):
        """Calcula y devuelve los parámetros. Lo implementa cada clase hija."""
        raise NotImplementedError("Cada clase hija debe implementar aprender().")

    def aplicar(self, df):
        """Aplica la transformación. Lo implementa cada clase hija."""
        raise NotImplementedError("Cada clase hija debe implementar aplicar().")


# La clase base no se usa sola
try:
    Transformador(COLUMNA_IMPUTAR).ajustar(datos)
except NotImplementedError as error:
    print("NotImplementedError:", error)

NotImplementedError: Cada clase hija debe implementar aprender().


In [15]:
class ImputadorMediana(Transformador):
    """Rellena los nulos con la mediana aprendida.

    El paréntesis (Transformador) es la HERENCIA: esta clase recibe todo lo
    que tiene Transformador sin volver a escribirlo. Solo implementa los dos
    métodos que le faltaban.
    """

    def aprender(self, df):
        return {"mediana": float(df[self.columna].median()),
                "nulos_en_ajuste": int(df[self.columna].isna().sum())}

    def aplicar(self, df):
        df[self.columna] = df[self.columna].fillna(self._parametros["mediana"])
        return df


class CodificadorNominal(Transformador):
    """Convierte una columna de categorías en columnas 0/1.

    El vocabulario se aprende en el ajuste: una categoría que solo aparece en
    prueba no genera columna nueva, porque el modelo no pudo aprender de ella.
    """

    def aprender(self, df):
        # Se guarda la lista de categorías ORDENADA, para que el resultado sea
        # el mismo en cada ejecución. Sin sorted(), el orden podría variar.
        return {"categorias": sorted(df[self.columna].dropna().unique())}

    def aplicar(self, df):
        # Se recorre el vocabulario aprendido, NO las categorías de este df.
        # Por eso una categoría nueva en prueba no genera columna: el modelo
        # nunca la vio y no podría haber aprendido nada de ella.
        for categoria in self._parametros["categorias"]:
            # Los nombres de columna no deben llevar espacios ni guiones
            etiqueta = str(categoria).strip().replace(" ", "_").replace("-", "_")
            # La comparación devuelve True/False; astype(int) lo pasa a 1/0
            df[f"{self.columna}_{etiqueta}"] = (df[self.columna] == categoria).astype(int)

        # La columna original ya no aporta: su información quedó en las nuevas
        return df.drop(columns=[self.columna])


class EscaladorEstandar(Transformador):
    """Centra en cero y escala a desviación uno."""

    def aprender(self, df):
        desviacion = float(df[self.columna].std())
        return {"media": float(df[self.columna].mean()),
                "desviacion": desviacion if desviacion != 0 else 1.0}

    def aplicar(self, df):
        df[self.columna] = ((df[self.columna] - self._parametros["media"])
                            / self._parametros["desviacion"])
        return df


class EliminadorColumnas(Transformador):
    """Quita columnas que no aportan al análisis, como el identificador."""

    def aprender(self, df):
        return {"existe": self.columna in df.columns}

    def aplicar(self, df):
        return df.drop(columns=[self.columna]) if self.columna in df.columns else df


imp = ImputadorMediana(COLUMNA_IMPUTAR)
salida = imp.ajustar_transformar(datos)
print(imp.nombre, "->", imp.parametros)
print(f"Nulos en {COLUMNA_IMPUTAR} después:", int(salida[COLUMNA_IMPUTAR].isna().sum()))

ImputadorMediana(actividad_fisica_cod) -> {'mediana': 5.0, 'nulos_en_ajuste': 1227}
Nulos en actividad_fisica_cod después: 0


### Comprobación de la herencia

La herencia se verifica directamente con `isinstance` y con la cadena MRO.

In [16]:
imputador = ImputadorMediana(COLUMNA_IMPUTAR)

# 1) El objeto ES un ImputadorMediana Y TAMBIÉN es un Transformador
print("¿Es ImputadorMediana?", isinstance(imputador, ImputadorMediana))
print("¿Es Transformador?   ", isinstance(imputador, Transformador))

# 2) La cadena de búsqueda muestra de dónde hereda
print("\nCadena de herencia:", [c.__name__ for c in ImputadorMediana.__mro__])

# 3) Métodos que NO están escritos en la clase hija y sin embargo funcionan,
#    porque se heredaron de la clase base
print("\nMétodos heredados de Transformador:")
for metodo in ["ajustar", "transformar", "ajustar_transformar"]:
    definido_en = "ImputadorMediana" if metodo in ImputadorMediana.__dict__ else "Transformador"
    print(f"  {metodo:<22} definido en {definido_en}")

print("\nMétodos propios de la clase hija:")
for metodo in ["aprender", "aplicar"]:
    print(f"  {metodo:<22} definido en ImputadorMediana")

¿Es ImputadorMediana? True
¿Es Transformador?    True

Cadena de herencia: ['ImputadorMediana', 'Transformador', 'object']

Métodos heredados de Transformador:
  ajustar                definido en Transformador
  transformar            definido en Transformador
  ajustar_transformar    definido en Transformador

Métodos propios de la clase hija:
  aprender               definido en ImputadorMediana
  aplicar                definido en ImputadorMediana


**Lo que muestra esta celda.** `ImputadorMediana` tiene ocho líneas de código propio y dispone
de todo el control de estado de la clase base. Si hay que cambiar un mensaje de error, se
cambia en un solo lugar y las cuatro clases hijas lo heredan.

**Polimorfismo** significa que quien usa estas clases no necesita saber de qué tipo es cada
una: todas ofrecen `ajustar` y `transformar`.

In [17]:
pasos = [
    EliminadorColumnas(COLUMNA_ID),
    ImputadorMediana(COLUMNA_IMPUTAR),
    CodificadorNominal("raceeth_cod"),
    CodificadorNominal("sexo_cod"),
    EscaladorEstandar(COLUMNA_ESCALAR),
]

resultado = datos.copy()
for paso in pasos:                      # la misma llamada para los cinco
    resultado = paso.ajustar_transformar(resultado)
    print(f"{paso.nombre:<34} -> {resultado.shape}")

EliminadorColumnas(id_registro)    -> (20103, 25)
ImputadorMediana(actividad_fisica_cod) -> (20103, 25)
CodificadorNominal(raceeth_cod)    -> (20103, 32)
CodificadorNominal(sexo_cod)       -> (20103, 33)
EscaladorEstandar(edad_cod)        -> (20103, 33)


**Este bloque es la evidencia de los tres principios**, y así lo declaramos en el
informe:

- **Herencia**: las cuatro clases heredan de `Transformador` y no repiten el control de estado.
- **Polimorfismo**: el bucle llama `ajustar_transformar()` sin preguntar el tipo.
- **Encapsulamiento**: `_parametros` y `_ajustado` son internos, y hay validación antes de actuar.

La consecuencia práctica: **agregar un transformador nuevo no obliga a
tocar el bucle**. Eso es bajo acoplamiento.

---
# 6. El `Pipeline`: componer los pasos

Una clase que guarda el orden, los ejecuta y controla que se haya ajustado antes de
transformar.

In [18]:
class Pipeline:
    """Encadena transformadores y los ejecuta en orden."""

    def __init__(self, pasos=None):
        self._pasos = list(pasos) if pasos else []
        self._ajustado = False

    def agregar(self, transformador):
        if not isinstance(transformador, Transformador):
            raise TypeError(
                f"Se esperaba un Transformador y se recibió {type(transformador).__name__}."
            )
        self._pasos.append(transformador)
        return self

    def ajustar(self, df):
        """Aprende los parámetros de cada paso SOLO con estos datos."""
        intermedio = df.copy()
        for paso in self._pasos:
            intermedio = paso.ajustar_transformar(intermedio)
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError("Pipeline: hay que ajustar antes de transformar.")
        resultado = df.copy()
        for paso in self._pasos:
            resultado = paso.transformar(resultado)
        return resultado

    def pasos_ejecutados(self):
        """Devuelve los pasos, para poder registrar sus parámetros aprendidos."""
        return tuple(self._pasos)

    def resumen(self):
        return pd.DataFrame([
            {"orden": i, "paso": p.nombre, "clase": type(p).__name__}
            for i, p in enumerate(self._pasos, start=1)
        ])

    def __len__(self):
        return len(self._pasos)

    def __repr__(self):
        estado = "ajustado" if self._ajustado else "sin ajustar"
        return f"Pipeline({len(self._pasos)} pasos, {estado})"


pipeline = Pipeline([
    EliminadorColumnas(COLUMNA_ID),
    ImputadorMediana(COLUMNA_IMPUTAR),
    CodificadorNominal("raceeth_cod"),
    CodificadorNominal("sexo_cod"),
    EscaladorEstandar(COLUMNA_ESCALAR),
    EscaladorEstandar(COLUMNA_IMPUTAR),
])

print(pipeline)
pipeline.resumen()

Pipeline(6 pasos, sin ajustar)


,orden,paso,clase
0,1,EliminadorColumnas(id_registro),EliminadorColumnas
1,2,ImputadorMediana(actividad_fisica_cod),ImputadorMediana
2,3,CodificadorNominal(raceeth_cod),CodificadorNominal
3,4,CodificadorNominal(sexo_cod),CodificadorNominal
4,5,EscaladorEstandar(edad_cod),EscaladorEstandar
5,6,EscaladorEstandar(actividad_fisica_cod),EscaladorEstandar


In [19]:
# Ajustar con entrenamiento, transformar prueba: sin fuga de datos
pipeline.ajustar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, " -> Prueba transformada:", prueba_lista.shape)
print(f"\nMedia de {COLUMNA_ESCALAR} en prueba tras escalar:", round(prueba_lista[COLUMNA_ESCALAR].mean(), 4))
print("No es exactamente cero, y está bien: los parámetros vienen del entrenamiento.")

Entrenamiento: (12564, 26)  -> Prueba transformada: (3141, 33)

Media de edad_cod en prueba tras escalar: -0.013
No es exactamente cero, y está bien: los parámetros vienen del entrenamiento.


**Por qué la media no da cero.** El escalador aprendió la media del entrenamiento. Si diera
exactamente cero sobre prueba, significaría que aprendió de datos que no debía ver. Este
resultado confirma que no hay fuga de datos.

---
# 7. Cohesión y acoplamiento

Son los dos conceptos centrales del criterio de diseño estructurado. Aquí los mostramos con
código.

**Alta cohesión** significa que cada componente hace **una sola cosa**.

**Bajo acoplamiento** significa que los componentes dependen poco unos de otros: cambiar uno no
obliga a tocar los demás.

## 7.1 Cohesión baja: una clase que hace de todo

Es el diseño que aparece cuando se escribe rápido: funciona, pero cada método toca cosas
distintas y la clase no tiene un propósito único.

In [20]:
class ProcesadorTodoEnUno:
    """Ejemplo de COHESIÓN BAJA: hace cuatro cosas sin relación entre sí."""

    def __init__(self, ruta):
        self.ruta = ruta
        self.df = None

    def cargar(self):            # responsabilidad 1: entrada y salida de archivos
        self.df = pd.read_csv(self.ruta)

    def limpiar(self):           # responsabilidad 2: transformar datos
        self.df = self.df.dropna()

    def graficar(self):          # responsabilidad 3: visualización
        pass

    def enviar_correo(self):     # responsabilidad 4: comunicación
        pass


print("Cuatro responsabilidades distintas en una sola clase.")
print("Problema práctico: para probar la limpieza hay que tener un archivo real,")
print("porque cargar() y limpiar() viven pegadas en el mismo objeto.")

Cuatro responsabilidades distintas en una sola clase.
Problema práctico: para probar la limpieza hay que tener un archivo real,
porque cargar() y limpiar() viven pegadas en el mismo objeto.


## 7.2 Cohesión alta: cada clase con un propósito

El mismo trabajo, repartido. Ahora cada pieza se puede probar por separado.

In [21]:
class Cargador:
    """Una sola responsabilidad: leer datos desde una fuente."""

    def __init__(self, ruta):
        self.ruta = ruta

    def cargar(self):
        return pd.read_csv(self.ruta)


class Limpiador:
    """Una sola responsabilidad: transformar un DataFrame que le entregan.

    NO sabe de dónde vienen los datos. Recibe un DataFrame y
    devuelve otro. Eso es bajo acoplamiento: no depende del Cargador.
    """

    def limpiar(self, df):
        return df.dropna()


# La ventaja se ve al probar: no hace falta ningún archivo
mini = pd.DataFrame({"a": [1.0, np.nan, 3.0], "b": [4.0, 5.0, 6.0]})
print("Se puede probar el Limpiador sin tocar el disco:")
print("  entrada:", mini.shape, " -> salida:", Limpiador().limpiar(mini).shape)

Se puede probar el Limpiador sin tocar el disco:
  entrada: (3, 2)  -> salida: (2, 2)


## 7.3 Prueba del acoplamiento

Usamos una pregunta concreta:

> **Si cambiamos la forma de imputar, ¿cuántos archivos hay que modificar?**

Si la respuesta es **uno**, el acoplamiento es bajo; si son varios, el diseño es frágil. Lo
comprobamos con el pipeline que ya construimos.

In [22]:
# Cambiar un paso del pipeline no obliga a tocar ni el Pipeline ni los otros pasos
pipeline_a = Pipeline([EliminadorColumnas(COLUMNA_ID), ImputadorMediana(COLUMNA_IMPUTAR)])
pipeline_b = Pipeline([EliminadorColumnas(COLUMNA_ID), ImputadorMediana("sueno_cod")])

for nombre, pipe in [("versión A", pipeline_a), ("versión B", pipeline_b)]:
    salida = pipe.ajustar(datos).transformar(datos)
    print(f"{nombre}: {salida.shape}")

print("\nSe cambió el paso y NO se modificó la clase Pipeline ni las demás clases.")
print("Eso es bajo acoplamiento.")

versión A: (20103, 25)
versión B: (20103, 25)

Se cambió el paso y NO se modificó la clase Pipeline ni las demás clases.
Eso es bajo acoplamiento.


## 7.4 Cómo se refleja en la estructura de archivos

Una organización con alta cohesión y bajo acoplamiento se reconoce a simple vista:

```
src/
├── carga.py            solo lee y verifica archivos
├── transformadores.py  solo las clases de transformación
├── pipeline.py         solo encadena y ejecuta
└── medicion.py         solo compara implementaciones
```

Cada archivo tiene un propósito que se enuncia en una frase. Si para describir un archivo hace
falta la palabra «y» varias veces, probablemente tenga cohesión baja.

**Para el informe.** En el apartado de diseño estructurado justificamos la organización con esta
prueba: «para cambiar la estrategia de imputación solo se modifica `transformadores.py`, porque
el pipeline no conoce el detalle de cada paso».

---
# 8. Validación: casos normales, límite y excepciones

Probamos los tres escenarios, uno de cada tipo, con el conjunto YRBS 2023 del proyecto.

In [23]:
# --- CASO NORMAL: el flujo completo sobre el conjunto real ---
pipe = Pipeline([EliminadorColumnas(COLUMNA_ID), ImputadorMediana(COLUMNA_IMPUTAR)])
salida = pipe.ajustar(datos).transformar(datos)

assert len(salida) == len(datos), "El pipeline perdió o duplicó filas"
assert salida[COLUMNA_IMPUTAR].isna().sum() == 0, f"Quedaron nulos en {COLUMNA_IMPUTAR}"
assert COLUMNA_ID not in salida.columns, "El identificador no se eliminó"
print("Caso normal: las tres comprobaciones pasaron.")

Caso normal: las tres comprobaciones pasaron.


In [24]:
# --- CASOS LÍMITE: situaciones extremas pero válidas ---

# 1) Una columna sin ningún nulo no debe alterarse
sin_nulos = pd.DataFrame({COLUMNA_IMPUTAR: [2.0, 5.0, 8.0]})
r1 = ImputadorMediana(COLUMNA_IMPUTAR).ajustar_transformar(sin_nulos)
assert r1[COLUMNA_IMPUTAR].tolist() == [2.0, 5.0, 8.0]
print("Límite 1: columna sin nulos, sin cambios.")

# 2) Una sola categoría genera una sola columna
una_cat = pd.DataFrame({"raceeth_cod": [5.0, 5.0, 5.0]})
r2 = CodificadorNominal("raceeth_cod").ajustar_transformar(una_cat)
print("Límite 2: una categoría ->", list(r2.columns))

# 3) Varianza cero no debe producir división por cero
constante = pd.DataFrame({COLUMNA_ESCALAR: [4.0, 4.0, 4.0]})
r3 = EscaladorEstandar(COLUMNA_ESCALAR).ajustar_transformar(constante)
assert r3[COLUMNA_ESCALAR].notna().all()
print("Límite 3: varianza cero ->", r3[COLUMNA_ESCALAR].tolist())

# 4) Una categoría nueva en prueba no debe crear columna
pipe_cat = Pipeline([CodificadorNominal("sexo_cod")]).ajustar(entrenamiento)
nuevos = prueba.copy()
nuevos.iloc[0, nuevos.columns.get_loc("sexo_cod")] = 9.0      # código que no existe en el codebook
r4 = pipe_cat.transformar(nuevos)
print("Límite 4: categoría nueva no crea columna ->",
      "sexo_cod_9.0" not in r4.columns)


Límite 1: columna sin nulos, sin cambios.
Límite 2: una categoría -> ['raceeth_cod_5.0']
Límite 3: varianza cero -> [0.0, 0.0, 0.0]
Límite 4: categoría nueva no crea columna -> True


In [25]:
# --- EXCEPCIONES: entradas que deben fallar con mensaje claro ---

pruebas = [
    ("transformar sin ajustar", lambda: ImputadorMediana(COLUMNA_IMPUTAR).transformar(datos)),
    ("columna inexistente", lambda: ImputadorMediana("no_existe").ajustar(datos)),
    ("agregar algo que no es Transformador", lambda: Pipeline().agregar("texto")),
    ("pipeline sin ajustar", lambda: Pipeline([ImputadorMediana(COLUMNA_IMPUTAR)]).transformar(datos)),
]

for descripcion, accion in pruebas:
    try:
        accion()
        print(f"  {descripcion:<38} NO lanzó excepción (revisar)")
    except (RuntimeError, KeyError, TypeError) as error:
        print(f"  {descripcion:<38} {type(error).__name__} capturado")

  transformar sin ajustar                RuntimeError capturado
  columna inexistente                    KeyError capturado
  agregar algo que no es Transformador   TypeError capturado
  pipeline sin ajustar                   RuntimeError capturado


Estos tres bloques (normal, límite y excepción) son la evidencia del criterio de
validación técnica.


## La verificación que más rinde

Si las clases producen **el mismo resultado** que la Fase 2, tenemos una prueba objetiva de
que la reorganización no rompió nada. La comparación se hace contra el **archivo que guardó la
Fase 2**, no contra código reescrito aquí: las 8 columnas `raza_*` de ese archivo se generaron
con `codificar_one_hot()` en F2, y el `CodificadorNominal` debe reproducirlas.


In [26]:
# Columnas one-hot que la Fase 2 guardó en su archivo (código -> nombre)
RAZA = {1: "raza_amerindia", 2: "raza_asiatica", 3: "raza_negra",
        4: "raza_hawaiana_pacifico", 5: "raza_blanca", 6: "raza_hispana",
        7: "raza_multiple_hispana", 8: "raza_multiple_no_hispana"}

# Versión Fase 3: el mismo trabajo con una clase
sin_raza_f2 = datos.drop(columns=list(RAZA.values()))       # se quitan las columnas de F2
v_f3 = CodificadorNominal("raceeth_cod").ajustar_transformar(sin_raza_f2)
v_f3 = v_f3.rename(columns={f"raceeth_cod_{float(c)}": nombre for c, nombre in RAZA.items()})

# En F2 la fila sin raza queda <NA> (no 0); se comparan las filas con respuesta
con_raza = datos["raceeth_cod"].notna()
f2 = datos.loc[con_raza, list(RAZA.values())].astype(int)
f3 = v_f3.loc[con_raza, list(RAZA.values())].astype(int)

pd.testing.assert_frame_equal(f2, f3)
assert datos.loc[~con_raza, list(RAZA.values())].isna().all().all()
print(f"Las 8 columnas raza_* de la Fase 2 coinciden con las de la clase en {con_raza.sum()} filas.")
print(f"Las {(~con_raza).sum()} filas sin raza siguen como NA en el archivo de F2, como se decidió.")
print("La reorganización no alteró nada.")


Las 8 columnas raza_* de la Fase 2 coinciden con las de la clase en 19733 filas.
Las 370 filas sin raza siguen como NA en el archivo de F2, como se decidió.
La reorganización no alteró nada.


---
# 9. Recursividad

Toda función recursiva tiene dos partes: un **caso base** que la detiene y un **caso
recursivo** que reduce el problema y vuelve a llamarse.

**Cuándo se justifica:** cuando no se sabe de antemano cuán profundo es el problema.
Si la profundidad es fija, un bucle es más simple y más rápido.

In [27]:
def cuenta_regresiva(n):
    """Ejemplo mínimo para ver las dos partes."""
    if n == 0:                       # CASO BASE: detiene la recursión
        print("Despegue")
        return
    print(n, end=" ")
    cuenta_regresiva(n - 1)          # CASO RECURSIVO


cuenta_regresiva(5)

5 4 3 2 1 Despegue


In [28]:
def aplanar(estructura, prefijo=""):
    """Convierte metadatos anidados en pares plano de clave y valor.

    Por qué recursión y no un bucle: la profundidad no se conoce al escribir el
    código. Mañana alguien agrega un nivel y el bucle anidado deja de servir.
    """
    plano = {}
    for clave in estructura:
        valor = estructura[clave]
        compuesta = f"{prefijo}.{clave}" if prefijo else str(clave)
        if isinstance(valor, dict) and valor:
            plano.update(aplanar(valor, compuesta))
        else:
            plano[compuesta] = valor
    return plano


metadatos = {
    "proyecto": {"nombre": "YRBS 2023 - redes sociales y salud mental", "fase": 3},
    "datos": {"filas": len(datos), "nulos": {COLUMNA_IMPUTAR: int(datos[COLUMNA_IMPUTAR].isna().sum())}},
    "entorno": {"semilla": SEMILLA, "librerias": {"pandas": pd.__version__}},
}

for clave, valor in aplanar(metadatos).items():
    print(f"{clave:<26} {valor}")

proyecto.nombre            YRBS 2023 - redes sociales y salud mental
proyecto.fase              3
datos.filas                20103
datos.nulos.actividad_fisica_cod 1227
entorno.semilla            42
entorno.librerias.pandas   3.0.6


## El costo de repetir trabajo

La recursión **no es** sinónimo de eficiencia. Las dos funciones siguientes siguen la misma idea
y tienen un costo muy distinto.

In [29]:
def fib_ingenua(n):
    """Sin memoria: recalcula los mismos subproblemas miles de veces."""
    if n < 2:
        return n
    return fib_ingenua(n - 1) + fib_ingenua(n - 2)


def fib_memoizada(n, cache=None):
    """La misma recursión, guardando lo ya calculado."""
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    resultado = n if n < 2 else fib_memoizada(n - 1, cache) + fib_memoizada(n - 2, cache)
    cache[n] = resultado
    return resultado


filas = []
for n in [18, 22, 26]:
    inicio = time.perf_counter(); fib_ingenua(n); t1 = time.perf_counter() - inicio
    inicio = time.perf_counter(); fib_memoizada(n); t2 = time.perf_counter() - inicio
    filas.append({"n": n, "ingenua_s": round(t1, 6), "memoizada_s": round(t2, 6),
                  "veces_mas_rapida": round(t1 / t2, 1)})

pd.DataFrame(filas)

,n,ingenua_s,memoizada_s,veces_mas_rapida
0,18,0.000708,0.000020,35.9
1,22,0.005223,0.000027,193.5
2,26,0.046047,0.000042,1088.6


El cambio entre una versión y otra es de dos líneas, y el efecto se mide en cientos de
veces. Lo que hace eficiente a la segunda no es la recursión: es **no repetir trabajo**.

**Decisión del proyecto.** En nuestro pipeline la recursión se justifica en un solo lugar:
`aplanar()`, porque los metadatos del proyecto son un diccionario anidado cuya profundidad
puede crecer. Los pasos de transformación tienen profundidad fija y se resuelven con bucles,
que son más simples y más rápidos.


---
# 10. Eficiencia: medir tiempo y memoria

Medimos de forma **reproducible** con `timeit` y `tracemalloc`, comparamos dos implementaciones
y las interpretamos considerando tiempo **y** memoria.

In [30]:
def con_bucle(df):
    """Clasifica las horas de sueño recorriendo las filas una por una.

    Códigos de q85: 1 = ≤4 h, 2 = 5 h, 3 = 6 h, 4 = 7 h, 5 = 8 h, 6 = 9 h, 7 = ≥10 h.
    """
    categorias = []
    for valor in df[COLUMNA_CLASIFICAR]:
        if pd.isna(valor):
            categorias.append("sin dato")
        elif valor < 3:
            categorias.append("muy insuficiente")   # ≤5 h
        elif valor < 5:
            categorias.append("insuficiente")       # 6-7 h
        else:
            categorias.append("suficiente")         # ≥8 h
    return categorias


def vectorizada(df):
    """Lo mismo, con una operación sobre la columna completa."""
    return pd.cut(df[COLUMNA_CLASIFICAR], bins=[-np.inf, 3, 5, np.inf],
                  labels=["muy insuficiente", "insuficiente", "suficiente"],
                  right=False).astype(object).fillna("sin dato").tolist()


# Antes de comparar: comprobar que dan el mismo resultado
assert con_bucle(datos) == vectorizada(datos), "Las versiones no coinciden"
print("Las dos implementaciones producen el mismo resultado.\n")

t_bucle = timeit.timeit(lambda: con_bucle(datos), number=20)
t_vect = timeit.timeit(lambda: vectorizada(datos), number=20)

print(f"Con bucle   : {t_bucle:.4f} s")
print(f"Vectorizada : {t_vect:.4f} s")
print(f"La vectorizada es {t_bucle / t_vect:.1f} veces más rápida")


Las dos implementaciones producen el mismo resultado.

Con bucle   : 0.2205 s
Vectorizada : 0.0500 s
La vectorizada es 4.4 veces más rápida


**La comprobación con `assert` va primero.** Una versión más rápida que entrega otro resultado
no es una optimización: es un error.

In [31]:
def medir(funcion, *args, **kwargs):
    """Ejecuta la función y devuelve resultado, segundos y memoria pico en MB.

    *args recoge los argumentos posicionales en una tupla y **kwargs los
    argumentos con nombre en un diccionario. El asterisco es lo que hace el
    trabajo; los nombres args y kwargs son solo convención.
    """
    tracemalloc.start()
    inicio = time.perf_counter()
    resultado = funcion(*args, **kwargs)
    transcurrido = time.perf_counter() - inicio
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return resultado, transcurrido, pico / 1024 / 1024


_, s_bucle, m_bucle = medir(con_bucle, datos)
_, s_vect, m_vect = medir(vectorizada, datos)

pd.DataFrame([
    {"version": "con bucle", "segundos": round(s_bucle, 5), "memoria_mb": round(m_bucle, 3)},
    {"version": "vectorizada", "segundos": round(s_vect, 5), "memoria_mb": round(m_vect, 3)},
])

,version,segundos,memoria_mb
0,con bucle,0.01421,0.165
1,vectorizada,0.00809,0.513


**Contraste tiempo–memoria.** La versión más rápida consume **más** memoria. Ese intercambio
entre tiempo y espacio es parte de lo que analizamos al elegir una implementación.

In [32]:
# Cómo crece el costo con el tamaño: esto es lo que revela la complejidad
mediciones = []
for n in [1000, 5000, 20000, 50000]:
    muestra = datos.sample(n=n, replace=True, random_state=SEMILLA)
    t_b = timeit.timeit(lambda: con_bucle(muestra), number=5) / 5
    t_v = timeit.timeit(lambda: vectorizada(muestra), number=5) / 5
    mediciones.append({"filas": n, "bucle_ms": round(t_b * 1000, 2),
                       "vectorizada_ms": round(t_v * 1000, 2),
                       "razon": round(t_b / t_v, 1)})

tabla = pd.DataFrame(mediciones)
print(tabla.to_string(index=False))
print("\nLa última columna es lo que importa: la ventaja crece con el tamaño.")

 filas  bucle_ms  vectorizada_ms  razon
  1000      0.46            1.10    0.4
  5000      2.87            1.71    1.7
 20000     11.95            3.23    3.7
 50000     29.74            5.79    5.1

La última columna es lo que importa: la ventaja crece con el tamaño.


**Un resultado a mirar con atención.** En los tamaños pequeños la versión vectorizada resulta
**igual o más lenta**. No es un error: `pd.cut` tiene un costo fijo de preparación que en mil
filas pesa más que el ahorro. La ventaja aparece cuando el volumen crece y ese costo fijo se
reparte entre más datos.

Por eso no basta con reportar qué versión ganó: hay que explicar **bajo qué condiciones**.

**Tres reglas que seguimos para que la medición valga**

1. **Repetir y conservar el tiempo menor**, no el promedio: los valores altos suelen
   reflejar interrupciones del sistema operativo, no el costo del código.
2. **Medir sobre el tamaño real** del conjunto. En mil filas todo es instantáneo.
3. **Comprobar que las versiones dan el mismo resultado.**

**Lo que buscamos:** que la medición **cambie una decisión**. Medir y no cambiar nada es un
ejercicio; medir, ver dónde está el costo y reescribir esa parte es optimización.


---
# 11. Patrones de diseño

Un patrón es una solución conocida a un problema que se repite. En la Fase 2
**ya usamos varios sin nombrarlos.**

## Strategy: varias formas de hacer lo mismo

En la Fase 2 comparamos cuatro formas de tratar los faltantes (conservar, eliminar filas,
moda y mediana), cada una escrita como un bloque distinto. Si cada forma es un bloque
distinto, cambiar de una a otra obliga a reescribir. Strategy separa el **qué** del **cómo**.


In [33]:
class EstrategiaImputacion:
    """Contrato común de todas las estrategias."""
    etiqueta = "sin definir"

    def calcular(self, df, columna):
        raise NotImplementedError


class PorMediana(EstrategiaImputacion):
    etiqueta = "mediana"

    def calcular(self, df, columna):
        return float(df[columna].median())


class PorMedia(EstrategiaImputacion):
    etiqueta = "media"

    def calcular(self, df, columna):
        return float(df[columna].mean())


class PorMedianaDeGrupo(EstrategiaImputacion):
    etiqueta = "mediana por grupo"

    def __init__(self, columna_grupo):
        self.columna_grupo = columna_grupo

    def calcular(self, df, columna):
        return df.groupby(self.columna_grupo)[columna].median().to_dict()


class ImputadorFlexible(Transformador):
    """No sabe imputar: sabe cuándo. El cómo lo aporta la estrategia."""

    def __init__(self, columna, estrategia=None):
        super().__init__(columna)          # llama al constructor de la clase base
        self.estrategia = estrategia or PorMediana()

    def aprender(self, df):
        return {"valor": self.estrategia.calcular(df, self.columna),
                "estrategia": self.estrategia.etiqueta}

    def aplicar(self, df):
        valor = self._parametros["valor"]
        if isinstance(valor, dict):                       # mediana por grupo
            relleno = df[self.estrategia.columna_grupo].map(valor)
            df[self.columna] = df[self.columna].fillna(relleno)
            df[self.columna] = df[self.columna].fillna(df[self.columna].median())
        else:
            df[self.columna] = df[self.columna].fillna(valor)
        return df


# Comparación de las tres estrategias en un solo bucle
desv_original = datos[COLUMNA_IMPUTAR].std()
comparacion = []
for estrategia in [PorMediana(), PorMedia(), PorMedianaDeGrupo(COLUMNA_GRUPO)]:
    salida = ImputadorFlexible(COLUMNA_IMPUTAR, estrategia).ajustar_transformar(datos)
    comparacion.append({
        "estrategia": estrategia.etiqueta,
        "desv_antes": round(desv_original, 4),
        "desv_despues": round(salida[COLUMNA_IMPUTAR].std(), 4),
        "cambio_pct": round((salida[COLUMNA_IMPUTAR].std() - desv_original) / desv_original * 100, 2),
    })

pd.DataFrame(comparacion)

,estrategia,desv_antes,desv_despues,cambio_pct
0,mediana,2.503,2.4254,-3.10
1,media,2.503,2.4254,-3.10
2,mediana por grupo,2.503,2.4319,-2.84


**Nota sobre estos números (YRBS).** `actividad_fisica_cod` es un código discreto de 1 a 8, por lo que
la mediana y la media no coinciden y cada estrategia reduce la dispersión en distinta medida. Es solo una
comparación didáctica: en el proyecto **no se imputa** esta variable (bitácora 2.5).

Esta tabla es la comparación que justifica la elección de estrategia. Toda imputación por un
valor central reduce la dispersión; lo que comparamos es **cuánto**, y esa cifra es el
argumento para elegir.

## Factory: decidir qué construir

La transformación que corresponde depende del rol de la variable. Factory concentra esa
decisión en un solo lugar, en vez de repetirla en cada cuaderno.

In [34]:
def crear_transformador(rol, columna, **parametros):
    """Devuelve el transformador que corresponde al rol analítico."""
    rol = rol.strip().lower()

    if rol == "continua":
        return ImputadorFlexible(columna, parametros.get("estrategia"))
    if rol == "nominal":
        return CodificadorNominal(columna)
    if rol == "escalar":
        return EscaladorEstandar(columna)
    if rol == "identificador":
        return EliminadorColumnas(columna)

    raise ValueError(
        f"Rol desconocido: '{rol}'. "
        "Roles válidos: continua, nominal, escalar, identificador."
    )


# El diccionario de variables de la Fase 1 pasa a gobernar el pipeline
diccionario = [
    {"variable": COLUMNA_ID, "rol": "identificador"},
    {"variable": COLUMNA_IMPUTAR, "rol": "continua"},
    {"variable": "raceeth_cod", "rol": "nominal"},
    {"variable": "sexo_cod", "rol": "nominal"},
    {"variable": COLUMNA_ESCALAR, "rol": "escalar"},
]

pipeline_auto = Pipeline([crear_transformador(v["rol"], v["variable"]) for v in diccionario])
salida = pipeline_auto.ajustar(datos).transformar(datos)

print(pipeline_auto.resumen().to_string(index=False))
print("\nResultado:", salida.shape)

try:
    crear_transformador("geografica", "estrato")
except ValueError as error:
    print("\nValueError:", error)

 orden                                    paso              clase
     1         EliminadorColumnas(id_registro) EliminadorColumnas
     2 ImputadorFlexible(actividad_fisica_cod)  ImputadorFlexible
     3         CodificadorNominal(raceeth_cod) CodificadorNominal
     4            CodificadorNominal(sexo_cod) CodificadorNominal
     5             EscaladorEstandar(edad_cod)  EscaladorEstandar

Resultado: (20103, 33)

ValueError: Rol desconocido: 'geografica'. Roles válidos: continua, nominal, escalar, identificador.


## Observer: registrar sin estorbar

La bitácora de decisiones la escribíamos a mano después de ejecutar, con el riesgo de que
quedara desactualizada. Con Observer, el pipeline **avisa** cada vez que termina un paso y quien
quiera registrar se suscribe.

In [35]:
class Bitacora:
    """Observador que acumula lo que ocurre y lo entrega como tabla."""

    def __init__(self):
        self.registros = []

    def notificar(self, paso, filas, columnas, segundos, parametros):
        self.registros.append({"paso": paso, "filas": filas, "columnas": columnas,
                               "segundos": round(segundos, 5), "parametros": parametros})

    def a_dataframe(self):
        return pd.DataFrame(self.registros)


class ReporteConsola:
    """Otro observador: imprime mientras ocurre."""

    def notificar(self, paso, filas, columnas, segundos, parametros):
        print(f"  {paso:<34} {filas:>6} filas, {columnas:>3} col  [{segundos:.4f}s]")


class PipelineObservable(Pipeline):
    """Hereda todo de Pipeline y agrega la capacidad de ser observado."""

    def __init__(self, pasos=None):
        super().__init__(pasos)
        self._observadores = []

    def suscribir(self, observador):
        self._observadores.append(observador)
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError("Pipeline: hay que ajustar antes de transformar.")
        resultado = df.copy()
        for paso in self._pasos:
            inicio = time.perf_counter()
            resultado = paso.transformar(resultado)
            transcurrido = time.perf_counter() - inicio
            for observador in self._observadores:
                observador.notificar(paso.nombre, len(resultado), resultado.shape[1],
                                     transcurrido, paso.parametros)
        return resultado


bitacora = Bitacora()
pipe = PipelineObservable([
    EliminadorColumnas(COLUMNA_ID),
    ImputadorFlexible(COLUMNA_IMPUTAR),
    CodificadorNominal("raceeth_cod"),
    EscaladorEstandar(COLUMNA_ESCALAR),
])
pipe.suscribir(bitacora).suscribir(ReporteConsola())

print("Ejecución del pipeline:")
salida = pipe.ajustar(datos).transformar(datos)

print("\nBitácora producida por la propia ejecución:")
bitacora.a_dataframe()[["paso", "filas", "columnas", "segundos"]]

Ejecución del pipeline:
  EliminadorColumnas(id_registro)     20103 filas,  25 col  [0.0022s]
  ImputadorFlexible(actividad_fisica_cod)  20103 filas,  25 col  [0.0018s]
  CodificadorNominal(raceeth_cod)     20103 filas,  32 col  [0.0128s]
  EscaladorEstandar(edad_cod)         20103 filas,  32 col  [0.0058s]

Bitácora producida por la propia ejecución:


,paso,filas,columnas,segundos
0,EliminadorColumnas(id_registro),20103,25,0.00220
1,ImputadorFlexible(actividad_fisica_cod),20103,25,0.00184
2,CodificadorNominal(raceeth_cod),20103,32,0.01280
3,EscaladorEstandar(edad_cod),20103,32,0.00576


**Lo importante.** La bitácora no se escribió a mano: se genera cada
vez que el pipeline corre, así que no puede quedar desactualizada.

## Singleton: una sola configuración

Rutas, semilla y umbrales se necesitan en varios módulos. Si cada uno construye su
propia configuración, dos partes del pipeline pueden usar semillas distintas sin que
nadie lo note.

**La advertencia:** Singleton introduce estado global, que complica las pruebas. Por eso
lo usamos solo para configuración y el resto de los parámetros se pasa explícitamente.

In [36]:
class Configuracion:
    """Una sola instancia compartida por todo el proyecto."""

    _instancia = None

    def __new__(cls, *args, **kwargs):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia._iniciada = False
        return cls._instancia

    def __init__(self, semilla=SEMILLA):
        if self._iniciada:               # evita reiniciar una instancia existente
            return
        self.semilla = semilla
        self.parametros = {}
        self._iniciada = True

    def definir(self, clave, valor):
        self.parametros[clave] = valor
        return self


a = Configuracion(semilla=42)
b = Configuracion(semilla=99)            # se ignora: la instancia ya existe
a.definir("umbral_nulos", 0.6).definir("test_size", 0.2)

print("¿Son el mismo objeto?", a is b)
print("Semilla vista desde b:", b.semilla)
print("Parámetros desde b   :", b.parametros)

¿Son el mismo objeto? True
Semilla vista desde b: 42
Parámetros desde b   : {'umbral_nulos': 0.6, 'test_size': 0.2}


## Resumen: qué patrón usamos y por qué

| Patrón | El problema que resuelve | Dónde aparece en nuestra Fase 2 |
|---|---|---|
| **Strategy** | Varias formas de hacer lo mismo | Comparar estrategias de faltantes (conservar, eliminar, moda, mediana) |
| **Factory** | Elegir qué construir según una condición | Decidir la transformación según el rol de la variable |
| **Observer** | Registrar sin que el ejecutor sepa quién anota | La bitácora de decisiones |
| **Singleton** | Una sola configuración compartida | La semilla y las rutas del proyecto |

**No usamos los cuatro en el proyecto.** El patrón principal es **Strategy**, aplicado al
tratamiento de faltantes; los otros tres se muestran como referencia. Un patrón aplicado donde
no hace falta agrega complejidad sin beneficio.


---
# 12. Pipeline construido desde la configuración

Hasta aquí los pasos se escribieron a mano, uno por uno. Esta sección construye el pipeline
**desde la celda de configuración**, de modo que al cambiar una columna no haya que tocar nada
más que esa celda.

Es Factory aplicado al proyecto completo: la configuración deja de ser documentación y pasa a
gobernar el código. Por eso también gobierna las decisiones de la Fase 2: con
`IMPUTAR_ORDINALES = False` y `ESCALAR_ORDINALES = False`, el pipeline conserva los NA y deja
las escalas ordinales tal como vienen.

In [37]:
def construir_pipeline(imputar=IMPUTAR_ORDINALES, escalar=ESCALAR_ORDINALES):
    """Arma el pipeline a partir de las columnas declaradas en la configuración.

    Orden de los pasos, y la razón de cada uno:
      1. eliminar el identificador, porque identifica y no describe
      2. imputar las ordinales, solo si la configuración lo pide
         (en el proyecto no: bitácora 2.5)
      3. codificar las nominales, porque un código 6 no «vale más» que un 2
      4. escalar las ordinales, solo si la configuración lo pide
         (en el proyecto no: sección 5 de F2)
    """
    pasos = []

    if COLUMNA_ID:
        pasos.append(EliminadorColumnas(COLUMNA_ID))

    if imputar:
        for columna in COLUMNAS_ORDINALES:
            pasos.append(ImputadorMediana(columna))

    for columna in COLUMNAS_NOMINALES:
        pasos.append(CodificadorNominal(columna))

    if escalar:
        for columna in COLUMNAS_ORDINALES:
            pasos.append(EscaladorEstandar(columna))

    return Pipeline(pasos)


pipeline_config = construir_pipeline()
print(f"Pipeline construido con {len(pipeline_config)} pasos desde la configuración")
print(f"(imputar = {IMPUTAR_ORDINALES}, escalar = {ESCALAR_ORDINALES}):\n")
print(pipeline_config.resumen().to_string(index=False))


Pipeline construido con 3 pasos desde la configuración
(imputar = False, escalar = False):

 orden                            paso              clase
     1 EliminadorColumnas(id_registro) EliminadorColumnas
     2 CodificadorNominal(raceeth_cod) CodificadorNominal
     3    CodificadorNominal(sexo_cod) CodificadorNominal


In [38]:
# Se ajusta con entrenamiento y se aplica a prueba, sin fuga de datos
pipeline_config.ajustar(entrenamiento)
entrenamiento_listo = pipeline_config.transformar(entrenamiento)
prueba_lista = pipeline_config.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, "->", entrenamiento_listo.shape)
print("Prueba       :", prueba.shape, "->", prueba_lista.shape)

# Las dos salidas deben tener exactamente las mismas columnas y en el mismo orden
assert list(entrenamiento_listo.columns) == list(prueba_lista.columns), \
    "Entrenamiento y prueba quedaron con columnas distintas"
print("\nAmbos conjuntos tienen las mismas columnas, en el mismo orden.")

Entrenamiento: (12564, 26) -> (12564, 33)
Prueba       : (3141, 26) -> (3141, 33)

Ambos conjuntos tienen las mismas columnas, en el mismo orden.


**Por qué importa esta comprobación.** Si el codificador aprendiera el vocabulario de cada
conjunto por separado, entrenamiento y prueba quedarían con columnas distintas y cualquier
modelo posterior fallaría. Como el vocabulario se aprende una sola vez en el ajuste, eso no
puede pasar.

## Guardar el resultado y su configuración

El conjunto procesado no sirve sin el registro de cómo se produjo; por eso guardamos ambos
juntos, lo que hace el trabajo trazable.

In [39]:
def guardar_resultado(df, ruta_salida, pipeline, carpeta=DIR_DEMO):
    """Escribe el conjunto procesado y un registro de los parámetros aprendidos.

    Se guarda en una carpeta de demostración: el dataset oficial del proyecto
    sigue siendo el de la Fase 2.
    """
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    destino = carpeta / ruta_salida
    df.to_csv(destino, index=False)

    registro = pd.DataFrame([
        {"orden": i, "paso": p.nombre, "parametros": str(p.parametros)}
        for i, p in enumerate(pipeline.pasos_ejecutados(), start=1)
    ])
    destino_param = destino.with_name(destino.stem + "_parametros.csv")
    registro.to_csv(destino_param, index=False)

    print(f"Conjunto guardado   : {destino.relative_to(RAIZ).as_posix()} "
          f"({df.shape[0]} filas x {df.shape[1]} col)")
    print(f"Parámetros guardados: {destino_param.relative_to(RAIZ).as_posix()}")
    return registro


registro = guardar_resultado(entrenamiento_listo, "demo_entrenamiento_config.csv",
                             pipeline_config)
registro.head()

Conjunto guardado   : F3/data/_demo/demo_entrenamiento_config.csv (12564 filas x 33 col)
Parámetros guardados: F3/data/_demo/demo_entrenamiento_config_parametros.csv


,orden,paso,parametros
0,1,EliminadorColumnas(id_registro),{'existe': True}
1,2,CodificadorNominal(raceeth_cod),"{'categorias': [np.float64(1.0), np.float64(2...."
2,3,CodificadorNominal(sexo_cod),"{'categorias': [np.float64(1.0), np.float64(2...."


Si cambia el conjunto, basta con editar la celda de configuración y volver a ejecutar el
cuaderno completo. Si una columna no coincide, el mensaje de error indica exactamente cuál.


---
# 13. El resultado: cómo queda el código y cómo quedan los datos

Esta sección cierra el recorrido. Muestra dos cosas: **cómo se ve el proyecto después de
aplicar POO** y **qué conjunto produce el pipeline** con las decisiones de la Fase 2.


## 13.1 El código, antes y después

**Antes (Fase 2).** El pipeline era una secuencia de celdas. Funcionaba, pero el orden
estaba implícito y nada se podía reutilizar:

```python
df = pd.read_csv(RUTA_DATOS)
df = df.drop(columns=["id_registro"])
df = pd.get_dummies(df, columns=["raceeth_cod", "sexo_cod"])
```

**Después (Fase 3).** El mismo trabajo, en cuatro líneas que se leen como una
declaración de intenciones:

```python
pipeline = construir_pipeline()
pipeline.ajustar(entrenamiento)
entrenamiento_listo = pipeline.transformar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)
```

Lo que cambió no es el resultado: es que ahora cada paso es una pieza con nombre, que se
puede probar, reemplazar y reutilizar sin tocar las demás.


## 13.2 La arquitectura, generada desde el propio código

En vez de escribir a mano la tabla de arquitectura del informe, la generamos leyendo las clases
que existen. Así no queda desactualizada.

In [40]:
def documentar_arquitectura():
    """Genera la tabla de arquitectura a partir de las clases definidas.

    __subclasses__() devuelve las clases que heredan de Transformador. Es la
    misma información que Python usa para resolver la herencia, así que la
    tabla refleja el código real y no lo que creemos que hay.
    """
    filas = [{
        "componente": "Transformador",
        "rol": "clase base",
        "responsabilidad": "Define el contrato y controla el estado de cada paso",
        "archivo sugerido": "src/transformadores.py",
    }]

    for clase in Transformador.__subclasses__():
        resumen = (clase.__doc__ or "sin documentar").strip().split("\n")[0]
        filas.append({
            "componente": clase.__name__,
            "rol": "clase hija",
            "responsabilidad": resumen,
            "archivo sugerido": "src/transformadores.py",
        })

    filas += [
        {"componente": "Pipeline", "rol": "orquestador",
         "responsabilidad": "Encadena los pasos y controla el orden de ajuste",
         "archivo sugerido": "src/pipeline.py"},
        {"componente": "PipelineObservable", "rol": "orquestador",
         "responsabilidad": "Pipeline que además notifica cada paso a sus observadores",
         "archivo sugerido": "src/pipeline.py"},
        {"componente": "Bitacora", "rol": "observador",
         "responsabilidad": "Registra lo ocurrido en cada paso de la ejecución",
         "archivo sugerido": "src/observadores.py"},
        {"componente": "construir_pipeline", "rol": "fábrica",
         "responsabilidad": "Arma el pipeline desde la celda de configuración",
         "archivo sugerido": "src/fabrica.py"},
        {"componente": "medir", "rol": "utilidad",
         "responsabilidad": "Mide tiempo y memoria de cualquier función",
         "archivo sugerido": "src/medicion.py"},
    ]
    return pd.DataFrame(filas)


arquitectura = documentar_arquitectura()
print(arquitectura.to_string(index=False))

        componente         rol                                                   responsabilidad       archivo sugerido
     Transformador  clase base              Define el contrato y controla el estado de cada paso src/transformadores.py
  ImputadorMediana  clase hija                       Rellena los nulos con la mediana aprendida. src/transformadores.py
CodificadorNominal  clase hija              Convierte una columna de categorías en columnas 0/1. src/transformadores.py
 EscaladorEstandar  clase hija                         Centra en cero y escala a desviación uno. src/transformadores.py
EliminadorColumnas  clase hija Quita columnas que no aportan al análisis, como el identificador. src/transformadores.py
 ImputadorFlexible  clase hija    No sabe imputar: sabe cuándo. El cómo lo aporta la estrategia. src/transformadores.py
          Pipeline orquestador                  Encadena los pasos y controla el orden de ajuste        src/pipeline.py
PipelineObservable orquestador         P

**Esa tabla es el apartado de documentación de arquitectura del informe**, junto con la
justificación de por qué el código se divide así.

La estructura de archivos que sugiere la última columna es la que proponemos para el
repositorio:

```
F3/
├── notebooks/
│   ├── S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb   este cuaderno (Formativa 3), ejecutado
│   └── S2_F3_NucleoAlgoritmico_POO_Grupo8.ipynb       pipeline real en clases (Sumativa 2)
├── src/
│   ├── transformadores.py          la clase base y sus hijas
│   ├── pipeline.py                 Pipeline y PipelineObservable
│   ├── observadores.py             Bitacora y ReporteConsola
│   ├── fabrica.py                  construir_pipeline()
│   └── medicion.py                 medir() y comparaciones
└── data/_demo/                     salidas de demostración de este cuaderno
```

## 13.3 El conjunto procesado: qué produce el pipeline

Aquí se ejecuta el pipeline completo, con las decisiones de la Fase 2 (NA conservados y
escalas ordinales sin escalar), y se revisa el resultado.


In [41]:
# Pipeline definitivo, construido desde la configuración
pipeline_final = construir_pipeline()
pipeline_final.ajustar(entrenamiento)

entrenamiento_final = pipeline_final.transformar(entrenamiento)
prueba_final = pipeline_final.transformar(prueba)

print("CONJUNTO PROCESADO")
print("=" * 58)
print(f"Entrenamiento : {entrenamiento_final.shape[0]:>6} filas x {entrenamiento_final.shape[1]:>3} columnas")
print(f"Prueba        : {prueba_final.shape[0]:>6} filas x {prueba_final.shape[1]:>3} columnas")
print(f"Partida desde : {datos.shape[0]:>6} filas x {datos.shape[1]:>3} columnas")
print(f"\nColumnas ganadas por la codificación: "
      f"{entrenamiento_final.shape[1] - datos.shape[1] + 1}")
entrenamiento_final.head(3)

CONJUNTO PROCESADO
Entrenamiento :  12564 filas x  33 columnas
Prueba        :   3141 filas x  33 columnas
Partida desde :  20103 filas x  26 columnas

Columnas ganadas por la codificación: 8


,peso_muestral,estrato,psu,edad_cod,salud_mental_cod,redes_sociales_cod,sueno_cod,actividad_fisica_cod,n_faltantes_analisis,caso_completo,...,raceeth_cod_1.0,raceeth_cod_2.0,raceeth_cod_3.0,raceeth_cod_4.0,raceeth_cod_5.0,raceeth_cod_6.0,raceeth_cod_7.0,raceeth_cod_8.0,sexo_cod_1.0,sexo_cod_2.0
9043,0.0172,300,572036,5.0,3.0,6.0,5.0,6.0,0,1,...,0,0,0,0,0,0,0,1,0,1
1704,0.7933,204,87717,6.0,3.0,6.0,4.0,8.0,0,1,...,0,0,0,0,1,0,0,0,1,0
4811,0.3088,300,372357,5.0,4.0,6.0,5.0,8.0,0,1,...,0,0,0,0,1,0,0,0,1,0


In [42]:
# Esquema final: qué tipo tiene cada columna y de dónde salió
def describir_esquema(df, originales):
    """Clasifica cada columna del resultado según su origen."""
    filas = []
    for columna in df.columns:
        if columna in originales:
            origen = "original"
        elif "_" in columna and columna.split("_")[0] in originales:
            origen = f"derivada de {columna.split('_')[0]}"
        else:
            origen = "derivada"
        filas.append({
            "columna": columna,
            "tipo": str(df[columna].dtype),
            "origen": origen,
            "nulos": int(df[columna].isna().sum()),
        })
    return pd.DataFrame(filas)


esquema = describir_esquema(entrenamiento_final, set(datos.columns))
print(f"Total de columnas: {len(esquema)}")
print(f"Originales conservadas: {(esquema['origen'] == 'original').sum()}")
print(f"Derivadas de la codificación: {(esquema['origen'] != 'original').sum()}")
print(f"Columnas con nulos: {(esquema['nulos'] > 0).sum()}")
print()
esquema.head(12)

Total de columnas: 33
Originales conservadas: 23
Derivadas de la codificación: 10
Columnas con nulos: 16



,columna,tipo,origen,nulos
0,peso_muestral,float64,original,0
1,estrato,int64,original,0
2,psu,int64,original,0
3,edad_cod,float64,original,55
4,salud_mental_cod,float64,original,0
5,redes_sociales_cod,float64,original,3281
6,sueno_cod,float64,original,724
7,actividad_fisica_cod,float64,original,487
8,n_faltantes_analisis,int64,original,0
9,caso_completo,int64,original,0


## 13.4 Verificación del resultado

Seis comprobaciones de que el pipeline respeta las decisiones de la Fase 2. Si alguna falla,
el problema es de esta fase y corregirlo aquí cuesta mucho menos que descubrirlo después.

In [43]:
def verificar_resultado(entrenamiento_final, prueba_final, datos_originales, objetivo):
    """Seis comprobaciones: el resultado respeta las decisiones de la Fase 2."""
    controles = []
    unidos = pd.concat([entrenamiento_final, prueba_final])

    # 1. No se perdieron ni se duplicaron filas
    esperadas = len(datos_originales)
    obtenidas = len(unidos)
    controles.append(("Se conservan todas las filas",
                      obtenidas == esperadas, f"{obtenidas} de {esperadas}"))

    # 2. Entrenamiento y prueba tienen el mismo esquema
    mismo_esquema = list(entrenamiento_final.columns) == list(prueba_final.columns)
    controles.append(("Mismo esquema en ambos conjuntos",
                      mismo_esquema, f"{entrenamiento_final.shape[1]} columnas"))

    # 3. Los faltantes se conservan: no se imputó (bitácora 2.5)
    nulos_antes = int(datos_originales[COLUMNAS_ORDINALES].isna().sum().sum())
    nulos_despues = int(unidos[COLUMNAS_ORDINALES].isna().sum().sum())
    controles.append(("Faltantes conservados como NA",
                      nulos_antes == nulos_despues, f"{nulos_despues} NA"))

    # 4. Las escalas ordinales siguen siendo códigos del codebook: no se escaló
    sin_escalar = all(
        set(unidos[c].dropna().unique()) <= set(datos_originales[c].dropna().unique())
        for c in COLUMNAS_ORDINALES)
    controles.append(("Escalas ordinales sin escalar", sin_escalar,
                      f"{len(COLUMNAS_ORDINALES)} columnas"))

    # 5. La variable objetivo sobrevivió intacta
    objetivo_ok = (objetivo in entrenamiento_final.columns
                   and entrenamiento_final[objetivo].isna().sum() == 0)
    controles.append(("Variable objetivo presente y completa",
                      objetivo_ok, objetivo))

    # 6. El identificador fue eliminado
    sin_id = COLUMNA_ID not in entrenamiento_final.columns
    controles.append(("Identificador eliminado", sin_id, COLUMNA_ID))

    tabla = pd.DataFrame(
        [{"control": c[0], "estado": "OK" if c[1] else "FALLA", "detalle": c[2]}
         for c in controles]
    )
    aprobado = all(c[1] for c in controles)
    return tabla, aprobado


controles, aprobado = verificar_resultado(entrenamiento_final, prueba_final,
                                          datos_objetivo, COLUMNA_OBJETIVO)
print(controles.to_string(index=False))
print("\n" + ("VERIFICACIÓN APROBADA: el resultado respeta las decisiones de la Fase 2."
               if aprobado else
               "VERIFICACIÓN RECHAZADA: revisar los controles en estado FALLA."))


                              control estado          detalle
         Se conservan todas las filas     OK   15705 de 15705
     Mismo esquema en ambos conjuntos     OK      33 columnas
        Faltantes conservados como NA     OK          5687 NA
        Escalas ordinales sin escalar     OK       4 columnas
Variable objetivo presente y completa     OK salud_mental_cod
              Identificador eliminado     OK      id_registro

VERIFICACIÓN APROBADA: el resultado respeta las decisiones de la Fase 2.


## 13.5 Salidas de demostración

Estos archivos documentan la ejecución de este cuaderno y se guardan en `F3/data/_demo/`, con
el prefijo `demo_`. **No reemplazan el dataset oficial** del proyecto, que sigue siendo
`F2/data/processed/yrbs2023_seleccion_procesada.csv`.

In [44]:
def guardar_salidas_demo(entrenamiento_final, prueba_final, pipeline, esquema,
                         carpeta=DIR_DEMO):
    """Escribe las cuatro salidas de demostración de este cuaderno."""
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    salidas = {}

    # 1. Los conjuntos procesados
    ruta_ent = carpeta / "demo_entrenamiento.csv"
    ruta_pru = carpeta / "demo_prueba.csv"
    entrenamiento_final.to_csv(ruta_ent, index=False)
    prueba_final.to_csv(ruta_pru, index=False)
    salidas["entrenamiento"] = ruta_ent
    salidas["prueba"] = ruta_pru

    # 2. El diccionario del conjunto resultante
    ruta_dic = carpeta / "demo_diccionario.csv"
    esquema.to_csv(ruta_dic, index=False)
    salidas["diccionario"] = ruta_dic

    # 3. Los parámetros aprendidos: es lo que permite reproducir el resultado
    ruta_par = carpeta / "demo_parametros.csv"
    pd.DataFrame([
        {"orden": i, "paso": p.nombre, "parametros": str(p.parametros)}
        for i, p in enumerate(pipeline.pasos_ejecutados(), start=1)
    ]).to_csv(ruta_par, index=False)
    salidas["parametros"] = ruta_par

    return salidas


salidas = guardar_salidas_demo(entrenamiento_final, prueba_final,
                               pipeline_final, esquema)

print("SALIDAS DE DEMOSTRACIÓN")
print("=" * 58)
for nombre, ruta in salidas.items():
    tamano = ruta.stat().st_size / 1024
    print(f"  {nombre:<16} {ruta.relative_to(RAIZ).as_posix():<36} {tamano:>7.1f} KB")

SALIDAS DE DEMOSTRACIÓN
  entrenamiento    F3/data/_demo/demo_entrenamiento.csv  1377.6 KB
  prueba           F3/data/_demo/demo_prueba.csv          345.0 KB
  diccionario      F3/data/_demo/demo_diccionario.csv       1.2 KB
  parametros       F3/data/_demo/demo_parametros.csv        0.3 KB


### Qué contiene cada salida

| Archivo | Qué contiene | Para qué sirve |
|---|---|---|
| `demo_entrenamiento.csv` | El conjunto de entrenamiento procesado por el pipeline | Revisar el resultado de las clases |
| `demo_prueba.csv` | El conjunto de prueba, con el mismo esquema | Comprobar que no hay fuga de datos |
| `demo_diccionario.csv` | Columna, tipo, origen y nulos | Saber qué significa cada variable |
| `demo_parametros.csv` | Lo que aprendió cada paso | Reproducir el resultado o auditarlo |

**La Fase 4 trabaja sobre el dataset de la Fase 2.** Si necesita una transformación nueva, la
agrega como un paso más del pipeline de clases, no como código suelto en su propio cuaderno.
Esa es la ventaja de haber modularizado.


In [45]:
# Comprobación final: lo que se guardó se puede volver a leer sin sorpresas
releido = pd.read_csv(salidas["entrenamiento"])

print("Verificación de ida y vuelta")
print(f"  Antes de guardar : {entrenamiento_final.shape}")
print(f"  Después de leer  : {releido.shape}")
print(f"  Mismas columnas  : {list(releido.columns) == list(entrenamiento_final.columns)}")
print(f"  Mismos NA        : {int(releido.isna().sum().sum()) == int(entrenamiento_final.isna().sum().sum())}")
print("\nLas salidas se pueden volver a leer sin cambios.")


Verificación de ida y vuelta
  Antes de guardar : (12564, 33)
  Después de leer  : (12564, 33)
  Mismas columnas  : True
  Mismos NA        : True

Las salidas se pueden volver a leer sin cambios.


---
# Cierre

## Resumen de conceptos

| Concepto | Dónde se ve en este cuaderno |
|---|---|
| Encapsulamiento | `_parametros`, `_ajustado`, propiedades de solo lectura |
| Herencia | `ImputadorMediana`, `CodificadorNominal` y `EscaladorEstandar` heredan de `Transformador` |
| Polimorfismo | El `Pipeline` recorre los pasos sin preguntar su tipo |
| Recursividad | `aplanar()` y el contraste entre Fibonacci ingenua y memoizada |
| Eficiencia | `timeit`, `perf_counter`, `tracemalloc` y la tabla de crecimiento |
| Patrones | Strategy (principal), Factory, Observer y Singleton |

## Conclusiones

- **Las clases no alteran los datos.** El `CodificadorNominal` reproduce las 8 columnas `raza_*`
  que guardó la Fase 2 (sección 8).
- **Separar `ajustar` de `transformar` evita la fuga de datos.** La media escalada en prueba no
  da cero porque los parámetros vienen solo del entrenamiento (sección 6).
- **La vectorización gana cuando el volumen crece.** En tamaños pequeños el costo fijo de
  `pd.cut` la hace igual o más lenta, y consume más memoria que el bucle (sección 10).
- **La recursión se justifica solo en `aplanar()`**, porque los metadatos tienen profundidad
  variable; el resto del pipeline usa bucles (sección 9).
- **El pipeline construido desde la configuración respeta las decisiones de la Fase 2:**
  conserva los faltantes como NA y no escala las escalas ordinales (sección 13).

## Bibliografía (APA 7)

Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1994). *Design patterns: Elements of
reusable object-oriented software*. Addison-Wesley.

McKinney, W. (2022). *Python for data analysis* (3.ª ed.). O'Reilly Media.

Python Software Foundation. (s. f.). *Classes*. https://docs.python.org/3/tutorial/classes.html

Python Software Foundation. (s. f.). *timeit — Measure execution time of small code
snippets*. https://docs.python.org/3/library/timeit.html

The pandas development team. (s. f.). *pandas documentation*. https://pandas.pydata.org/docs/

---

*Avance Fase 3 · Semana 2 · Formativa 3 · Grupo 8 · MCDI500 · Magíster en Ciencia de Datos e
Inteligencia Artificial · Universidad Andrés Bello*